In [1]:
import pandas as pd
import numpy as np
import os

In [2]:
data=pd.read_csv("pressure_analysis.csv")
data.head()

,match_id,innings,overid,over_ball_num,innings_ball_num,batting_team_id,bowling_team_id,striker_id,striker_name,non_striker_id,...,second_inn_revised_overs,revised_target,score_before,wickets_before,max_overs,balls_remaining,target,runs_needed,required_rr,pressure
0,13511,1,1,1,1,22,28,1187,Jason Roy,6273,...,NaN,NaN,0,0,NaN,119.0,101.0,101.0,0.0,False
1,13511,1,1,2,2,22,28,1187,Jason Roy,6273,...,NaN,NaN,0,0,NaN,118.0,101.0,101.0,0.0,False
2,13511,1,1,3,3,22,28,1187,Jason Roy,6273,...,NaN,NaN,0,0,NaN,117.0,101.0,101.0,0.0,False
3,13511,1,1,4,4,22,28,1187,Jason Roy,6273,...,NaN,NaN,0,0,NaN,116.0,101.0,101.0,0.0,False
4,13511,1,1,5,5,22,28,1187,Jason Roy,6273,...,NaN,NaN,0,0,NaN,115.0,101.0,101.0,0.0,False


# Check required columns

In [3]:
required = [
    "striker_id", "striker_name", "match_id",
    "runs", "is_wicket", "dismissed_player_id",
    "extras_type", "pressure"
]

In [5]:
for col in required:
    if col not in data.columns:
        raise ValueError(f"Missing column: {col}")


# Clean data

In [6]:
data["runs"] = pd.to_numeric(data["runs"], errors="coerce").fillna(0)

data["extras_type"] = (
    data["extras_type"].fillna("").astype(str).str.lower()
)

data["is_wicket"] = (
    data["is_wicket"].astype(str).str.lower()
    .isin(["true", "1", "yes"])
)

data["pressure"] = (
    data["pressure"].astype(str).str.lower()
    .isin(["true", "1", "yes"])
)

# Batter does not face a ball for a wide

In [8]:
data["batter_ball"] = (
    ~data["extras_type"].str.contains("wide", na=False)
).astype(int)

# Count dismissal only if the striker was dismissed

In [9]:
data["striker_out"] = (
    data["is_wicket"]
    & data["dismissed_player_id"].notna()
    & (
        pd.to_numeric(data["dismissed_player_id"], errors="coerce")
        == pd.to_numeric(data["striker_id"], errors="coerce")
    )
).astype(int)

# Calculate overall performance

In [11]:
overall = data.groupby(
    ["striker_id", "striker_name"]
).agg(
    overall_runs=("runs", "sum"),
    overall_balls=("batter_ball", "sum"),
    overall_outs=("striker_out", "sum")
).reset_index()

In [18]:
overall["overall_sr"] = (
    100 * overall["overall_runs"] /
    overall["overall_balls"].replace(0, np.nan)
)

In [19]:
overall["overall_average"] = (
    overall["overall_runs"] /
    overall["overall_outs"].replace(0, np.nan)
)

# Calculate pressure performance

In [20]:
p = data[data["pressure"]].copy()

In [21]:
pressure = p.groupby(
    ["striker_id", "striker_name"]
).agg(
    pressure_runs=("runs", "sum"),
    pressure_balls=("batter_ball", "sum"),
    pressure_matches=("match_id", "nunique"),
    pressure_outs=("striker_out", "sum")
).reset_index()

In [22]:
pressure["pressure_sr"] = (
    100 * pressure["pressure_runs"] /
    pressure["pressure_balls"].replace(0, np.nan)
)

In [23]:
pressure["pressure_average"] = (
    pressure["pressure_runs"] /
    pressure["pressure_outs"].replace(0, np.nan)
)

# Compare pressure performance with overall performance

In [24]:
result = pressure.merge(
    overall,
    on=["striker_id", "striker_name"],
    how="left"
)

In [25]:
result["sr_uplift"] = (
    result["pressure_sr"] - result["overall_sr"]
)


In [26]:
result["average_uplift"] = (
    result["pressure_average"] - result["overall_average"]
)

# Minimum eligibility requirements

In [27]:
result = result[
    (result["pressure_balls"] >= 30) &
    (result["pressure_matches"] >= 5)
].copy()


# Calculate standard scores

In [28]:
def zscore(series):
    sd = series.std(ddof=0)

    if pd.isna(sd) or sd == 0:
        return pd.Series(0.0, index=series.index)

    return (series - series.mean()) / sd

In [29]:
result["sr_z"] = zscore(result["pressure_sr"])
result["avg_z"] = zscore(result["pressure_average"])
result["sr_uplift_z"] = zscore(result["sr_uplift"])
result["avg_uplift_z"] = zscore(result["average_uplift"])

# Weighted batter score

In [30]:
raw_score = (
    0.45 * result["sr_z"] +
    0.25 * result["avg_z"] +
    0.20 * result["sr_uplift_z"] +
    0.10 * result["avg_uplift_z"]
)

# Confidence adjustment

In [31]:
confidence = (
    result["pressure_balls"] /
    (result["pressure_balls"] + 40)
) * (
    result["pressure_matches"] /
    (result["pressure_matches"] + 3)
)

In [32]:
result["rating"] = raw_score * confidence

# Rank batters

In [33]:
result = result.sort_values(
    ["rating", "pressure_balls"],
    ascending=[False, False]
).reset_index(drop=True)

result.insert(0, "rank", range(1, len(result) + 1))

# Save results

In [39]:
os.makedirs("output", exist_ok=True)

result.head(10).to_csv(
    "output/top_10_batters.csv",
    index=False
)

print("\nTOP 10 BATTERS")
print(
    result[
        ["rank", "striker_name", "pressure_runs",
         "pressure_balls", "pressure_sr",
         "pressure_average", "rating"]
    ].head(10).to_string(index=False)
)

print("\nBatter ranking completed!")
print("Saved:top_10_batters.csv")


TOP 10 BATTERS
 rank         striker_name  pressure_runs  pressure_balls  pressure_sr  pressure_average   rating
    1 Vaibhav Sooryavanshi            243              95   255.789474         60.750000 1.738414
    2     Suryakumar Yadav           1796             958   187.473904         42.761905 1.314862
    3       Shashank Singh            794             440   180.454545         46.705882 1.260643
    4         Rishabh Pant            754             407   185.257985         31.416667 1.206799
    5           Babar Azam            421             243   173.251029         42.100000 1.162326
    6      Reeza Hendricks            580             317   182.965300         38.666667 1.139946
    7          Virat Kohli            944             530   178.113208         41.043478 1.113475
    8     Liam Livingstone            890             472   188.559322         26.969697 1.063056
    9        Glenn Maxwell            704             378   186.243386         30.608696 1.040082
   1